# General Cleaning of Data for schools
Main idea is to get geospatial boundaries for each primary school every year, inclusive of any relocations/changes in boundaries. 

## Notebook Scope
This notebook constructs the school-side inputs used in the project. It starts from the general school register, adds coordinates, expands schools over time, incorporates manual relocation and opening-year adjustments, and then builds the school-boundary files used for spatial joins.


In [ ]:
import pandas as pd

df_schools=pd.read_csv('../data/raw/schools/Generalinformationofschools.csv')
df_schools.head()

To obtain all relocations, we are going to have to scrape the MOE website for all the news releases regarding school relocations, and then extract the relevant information (school name, year of relocation, new location, etc.) from those news releases. <br>
This is normally announced in their press release on start of Primary 1 Registration. 

In [ ]:
import csv
import gzip
import io
import re
from datetime import datetime
from typing import List, Optional, Set, Tuple

import requests
import xml.etree.ElementTree as ET

BASE = "https://www.moe.gov.sg"
ROBOTS_URL = f"{BASE}/robots.txt"
START_YEAR = 2014
END_YEAR = 2026

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; url-only-scraper/1.0)",
    "Accept-Language": "en-SG,en;q=0.9",
})

# URL slug patterns (strict -> broader)
SLUG_PATTERNS = [
    # Most common modern style
    re.compile(r"/news/press-releases/\d{8}-\d{4}-primary-one-registration-exercise-", re.I),
    re.compile(r"/news/press-releases/\d{8}-primary-one-registration-exercise-", re.I),

    # Variations that may appear in older years
    re.compile(r"/news/press-releases/\d{8}-\d{4}-p1-registration-exercise-", re.I),
    re.compile(r"/news/press-releases/\d{8}-p1-registration-exercise-", re.I),

    # Looser fallback (use if strict patterns find nothing)
    re.compile(r"/news/press-releases/\d{8}-.*primary-one-registration", re.I),
    re.compile(r"/news/press-releases/\d{8}-.*p1-registration", re.I),
]

def parse_date_from_slug(url: str) -> Optional[datetime]:
    slug = url.rstrip("/").split("/")[-1]
    m = re.match(r"(?P<y>\d{4})(?P<m>\d{2})(?P<d>\d{2})-", slug)
    if not m:
        return None
    try:
        return datetime(int(m.group("y")), int(m.group("m")), int(m.group("d")))
    except ValueError:
        return None

def slug_to_title(url: str) -> str:
    """
    Convert '20240516-2024-primary-one-registration-exercise-to-start-from-2-july-2024'
    -> '2024 primary one registration exercise to start from 2 july 2024' (title-cased nicely)
    """
    slug = url.rstrip("/").split("/")[-1]
    # remove the YYYYMMDD-
    slug = re.sub(r"^\d{8}-", "", slug)
    # hyphens to spaces
    words = slug.replace("-", " ").strip()
    # lightweight title casing (keep acronyms/numbers as-is)
    # You can swap to .title() if you prefer.
    return " ".join(w.upper() if w.lower() in {"moe", "p1"} else w for w in words.split())

def is_press_release(url: str) -> bool:
    return bool(re.search(r"^https://(www\.)?moe\.gov\.sg/news/press-releases/", url))

def url_matches(url: str) -> bool:
    return any(p.search(url) for p in SLUG_PATTERNS)

def get_sitemaps_from_robots() -> List[str]:
    r = session.get(ROBOTS_URL, timeout=30)
    r.raise_for_status()
    sitemaps = []
    for line in r.text.splitlines():
        if line.lower().startswith("sitemap:"):
            sitemaps.append(line.split(":", 1)[1].strip())
    return sitemaps or [f"{BASE}/sitemap.xml"]

def fetch_sitemap_bytes(url: str) -> bytes:
    r = session.get(url, timeout=60)
    r.raise_for_status()
    if url.endswith(".gz"):
        with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as gz:
            return gz.read()
    return r.content

def parse_sitemap(xml_bytes: bytes) -> Tuple[List[str], List[str]]:
    root = ET.fromstring(xml_bytes)
    tag = root.tag.split("}", 1)[-1]

    sitemap_links, url_links = [], []
    if tag == "sitemapindex":
        for loc in root.findall(".//{*}sitemap/{*}loc"):
            if loc.text:
                sitemap_links.append(loc.text.strip())
    elif tag == "urlset":
        for loc in root.findall(".//{*}url/{*}loc"):
            if loc.text:
                url_links.append(loc.text.strip())
    return sitemap_links, url_links

def crawl_all_urls(start_sitemaps: List[str], max_sitemaps: int = 2000) -> Set[str]:
    seen, out = set(), set()
    queue = list(start_sitemaps)

    while queue and len(seen) < max_sitemaps:
        sm = queue.pop(0)
        if sm in seen:
            continue
        seen.add(sm)
        try:
            xml_bytes = fetch_sitemap_bytes(sm)
            more_sitemaps, urls = parse_sitemap(xml_bytes)
            out.update(urls)
            for nxt in more_sitemaps:
                if nxt not in seen:
                    queue.append(nxt)
        except Exception as e:
            print(f"[warn] failed sitemap {sm}: {e}")

    return out

def main():
    sitemaps = get_sitemaps_from_robots()
    all_urls = crawl_all_urls(sitemaps)

    press_release_urls = [u for u in all_urls if is_press_release(u)]

    # Year filter from slug date
    dated = []
    for u in press_release_urls:
        dt = parse_date_from_slug(u)
        if dt and START_YEAR <= dt.year <= END_YEAR:
            dated.append(u)

    # URL-only keyword filter
    matched = [u for u in dated if url_matches(u)]

    # Write CSV from URL slug only
    out_csv = f"../data/processed/schools/moe_p1_press_releases_urlmatched_{START_YEAR}_{END_YEAR}.csv"
    rows = []
    for u in sorted(matched):
        dt = parse_date_from_slug(u)
        rows.append({
            "published_on": dt.strftime("%Y-%m-%d") if dt else "",
            "title_from_url": slug_to_title(u),
            "url": u,
        })

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["published_on", "title_from_url", "url"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Press releases in range: {len(dated)}")
    print(f"URL-matched P1 candidates: {len(matched)}")
    print(f"Wrote: {out_csv}")

    # Debug hint if 0
    if len(matched) == 0:
        print("\nNo matches. Try loosening patterns further, or print a few dated URLs to inspect:")
        print("Sample dated press release URLs:")
        for u in sorted(dated)[:20]:
            print(" ", u)

if __name__ == "__main__":
    main()

# 

## Filter to Primary Schools
These cells reduce the full school register to primary schools only, which is the relevant population for the school-exposure measures in the housing analysis.


In [ ]:
# filter out from school_name for primary schools
pri_schs=df_schools[df_schools['mainlevel_code'].str.contains('PRIMARY', case=False, na=False)]

In [ ]:
pri_schs['school_name'].unique()

## Geocode School Locations
This section queries OneMap by postal code to recover latitude-longitude and SVY21 coordinates for each primary school. Those point coordinates are the base geometry before yearly expansions and boundary matching.


In [ ]:
import pandas as pd
import requests
import time
from typing import Optional, Tuple
# Query OneMapAPI for Coordinates, lat long is EPSG:4326, EPSG:XY is 3414
def get_coordinates(postal_code: str) -> Optional[Tuple[float, float, float, float]]:
    url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={postal_code}&returnGeom=Y&getAddrDetails=N"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('found', 0) > 0:
                lat = data['results'][0]['LATITUDE']
                lon = data['results'][0]['LONGITUDE']
                X, Y = data['results'][0]['X'], data['results'][0]['Y']
                return float(lat), float(lon), float(X), float(Y)
        elif response.status_code in [429, 403]:
            # simple rate limit handling
            print("Rate limit exceeded or forbidden. Waiting for 10 seconds...")
            time.sleep(10)
            return get_coordinates(postal_code)  
        else:
            print(f"Error fetching data for {postal_code}: {response.status_code}")
    except Exception as e:
        print(f"Exception for {postal_code}: {e}")
    return None, None, None, None



In [ ]:
import tqdm

# Query coordinates for each school
pri_schs['latitude'] = None
pri_schs['longitude'] = None
pri_schs['X'] = None
pri_schs['Y'] = None

for idx, row in tqdm.tqdm(pri_schs.iterrows(), total=pri_schs.shape[0]):
    postal_code = row['postal_code']
    lat, lon, X, Y = get_coordinates(postal_code)
    pri_schs.at[idx, 'latitude'] = lat
    pri_schs.at[idx, 'longitude'] = lon
    pri_schs.at[idx, 'X'] = X
    pri_schs.at[idx, 'Y'] = Y

pri_schs.head()

## Expand Schools Across Years
The next cells duplicate schools across calendar years so the school panel can reflect when each campus is active. This gives the later joins a year-specific school roster instead of a single static snapshot.


In [ ]:
# Add a year column and duplicate rows for each year from 2014 to 2026
years = list(range(2014, 2027))
expanded_rows = []
for _, row in pri_schs.iterrows():
    for year in years:
        new_row = row.copy()
        new_row['year'] = year
        expanded_rows.append(new_row)
pri_schs_expanded = pd.DataFrame(expanded_rows)
pri_schs_expanded.head()

## Manual Relocation and Closure Adjustments
Some schools require hand-coded fixes for relocations or temporary campuses. These cells define those exceptions and append the relevant historical rows before saving an intermediate expanded file.


In [ ]:
d={
    'school_name': ['QIAONAN PRIMARY SCHOOL'.upper(), 'BEDOK WEST PRIMARY SCHOOL'.upper(), 'HONG KAH PRIMARY SCHOOL'.upper(), 'Coral Primary School'.upper(),'Balestier Hill Primary School'.upper(), 'Da Qiao Primary School'.upper(), 'EAST COAST PRIMARY SCHOOL'.upper(), 'East View Primary School'.upper(), 'Loyang Primary School'.upper(), 'MacPherson Primary School'.upper(),'Juying Primary School'.upper(), 'Eunos Primary School'.upper(), 'Stamford Primary School'.upper(), 'Guangyang Primary School'.upper()],
    'postal_code': [529454, 479225, 659203, 518902, 329927, 569185, 469031, 528907, 519419, 387724, 649037, 419529, 198423, 579806],
    'end_year': [2014, 2014, 2014, 2017, 2017, 2017, 2017, 2017, 2017, 2017, 2021, 2021, 2021, 2021]
}
temp_df=pd.DataFrame(d)
temp_df


In [ ]:
# Add geospatial coordinates for the schools in temp_df
temp_df['latitude'] = None
temp_df['longitude'] = None
temp_df['X'] = None
temp_df['Y'] = None
for idx, row in tqdm.tqdm(temp_df.iterrows(), total=temp_df.shape[0]):
    postal_code = row['postal_code']
    lat, lon, X, Y = get_coordinates(postal_code)
    temp_df.at[idx, 'latitude'] = lat
    temp_df.at[idx, 'longitude'] = lon
    temp_df.at[idx, 'X'] = X
    temp_df.at[idx, 'Y'] = Y

In [ ]:
# only duplicate the years from 2014 up to the year_to_stop for each school
expanded_rows = []
for _, row in temp_df.iterrows():
    for year in range(2014, row['year_to_stop'] + 1):
        new_row = row.copy()
        new_row['year'] = year
        expanded_rows.append(new_row)
merged_schs=pd.DataFrame(expanded_rows)
merged_schs.drop(columns=['year_to_stop'], inplace=True)
pri_schs_expanded_1 = pd.concat([pri_schs_expanded, merged_schs], ignore_index=True)

In [ ]:
pri_schs_expanded_1.info()

In [ ]:
pri_schs_expanded_1.to_csv('../data/processed/schools/primary_schools_expanded_without_new_schs.csv', index=False)

## Opening-Year Adjustments
Newly opened schools should only appear from their opening year onward. This section removes pre-opening rows so the final annual school panel respects those start dates.


# School open: 
- 2014: 
    - West Spring Primary School
    - Alexandra Primary School
    - Northoaks Primary School

- 2015:
    - Angsana Primary School

- 2016:
    - Oasis Primary School
    - Punggol Cove Primary School
    - Waterway Primary School

- 2018: 
    - Fern Green Primary School

- 2020: 
    - Valour Primary School

- 2021:
    - Northshore Primary School


In [ ]:
d1={
    2015: ['AGSANA PRIMARY SCHOOL'],
    2016: ['OASIS PRIMARY SCHOOL', 'PUNGGOL COVE PRIMARY SCHOOL', 'WATERWAY PRIMARY SCHOOL'],
    2018: ['FERN GREEN PRIMARY SCHOOL'],
    2020: ['VALOUR PRIMARY SCHOOL'],
    2021: ['NORTHSHORE PRIMARY SCHOOL']
}
# pri_schs_expanded_2=pri_schs_expanded_1.copy()
# any year before the year should be deleted for that school, but the school should be kept for that year and all years after
# for year, schools in d1.items():
#     pri_schs_expanded_2 = pri_schs_expanded_2[~((pri_schs_expanded_2['school_name'].isin(schools)) & (pri_schs_expanded_2['year'] < year))]

for year, schools in d1.items():
    mask = pri_schs_expanded_1['school_name'].isin(schools)
    pri_schs_expanded_1.loc[mask, 'start_year'] = year

In [ ]:
pri_schs_expanded_1.reset_index(drop=True, inplace=True)

In [ ]:
pri_schs_expanded_1.to_csv('../data/processed/schools/final_primary_schools.csv', index=False)

## Build School Boundary Files
The remainder of the notebook shifts from point locations to boundary geometries. It matches yearly school records to land-use polygons, checks difficult cases manually, and exports the final boundary-based school files used in downstream processing.


## 13/3 - Adding boundaries to the school, instead of the center

In [ ]:
import geopandas as gpd
from pathlib import Path

# def resolve_data_path(relative_path: str) -> Path:
#     candidates = [Path(relative_path), Path('..') / relative_path]
#     for path in candidates:
#         if path.exists():
#             return path
#     raise FileNotFoundError(f'Could not find {relative_path}')

def load_education_land_use(path: Path) -> gpd.GeoDataFrame:
    land_use = gpd.read_file(path)

    if land_use.crs is None:
        raise ValueError(f"{path} has no CRS defined")
    if land_use.crs.to_epsg() != 3414:
        land_use = land_use.to_crs(epsg=3414)

    # normalize description column
    land_use['LU_DESC'] = land_use['LU_DESC'].astype(str).str.strip()

    # keep education-related land use
    education_pattern = r'EDUCATION|SCHOOL|COLLEGE|UNIVERSITY|POLYTECHNIC|INSTITUTE'

    land_use = land_use[
        land_use['LU_DESC'].str.contains(education_pattern, case=False, na=False)
    ].copy()

    land_use = land_use.reset_index(drop=True).reset_index(names='land_use_index')
    return land_use

LAND_USE_SOURCES = {
    'mp2014': {
        'path': '../data/raw/land_use/2014/G_MP14_LAND_USE_PL.shp',
        'years': range(2014, 2019),
    },
    'mp2019': {
        'path': '../data/raw/land_use/MasterPlan2019LandUselayer.geojson',
        'years': range(2019, 2025),
    },
    'mp2025': {
        'path': '../data/raw/land_use/MasterPlan2025LandUseLayer.geojson',
        'years': range(2025, 2027),
    },
}

land_use_layers = {
    source_name: load_education_land_use(config['path'])
    for source_name, config in LAND_USE_SOURCES.items()
}

for source_name, land_use_layer in land_use_layers.items():
    print(source_name, land_use_layer.shape)


In [ ]:
valid={
    2018: 'mp2014',
    2024: 'mp2019',
    2026: 'mp2025'
}

In [ ]:
import pandas as pd

# Read primary schools data and keep only years covered by the land-use layers
pri_schs_df = pd.read_csv('../data/processed/schools/final_primary_schools.csv')

In [ ]:
pri_schs_df.drop(columns=['latitude', 'longitude'], inplace=True)

In [ ]:
keys = sorted(valid.keys())

def assign_land_use(end_year):
    for key in keys:
        if end_year <= key:
            return valid[key]
    return pd.NA

pri_schs_df['land_use_source'] = pri_schs_df['end_year'].apply(assign_land_use)

In [ ]:
gdf_schools = gpd.GeoDataFrame(
    pri_schs_df,
    geometry=gpd.points_from_xy(pri_schs_df['X'], pri_schs_df['Y']),
    crs="EPSG:3414"
)

def attach_school_boundaries(
    schools: gpd.GeoDataFrame,
    land_use_sources: dict,
    land_use_layers: dict,
) -> gpd.GeoDataFrame:
    matched_parts = []

    for source_name, config in land_use_sources.items():
        school_subset = schools[schools['land_use_source'] == source_name].copy()
        joined = school_subset.sjoin(
            land_use_layers[source_name][['land_use_index', 'LU_DESC', 'geometry']],
            how='left',
            predicate='within'
        ).rename(columns={'geometry': 'school_point'})

        joined = joined.merge(
            land_use_layers[source_name][['land_use_index', 'geometry']],
            on='land_use_index',
            how='left'
        )

        matched_parts.append(joined)

    school_boundaries = pd.concat(matched_parts, ignore_index=True)
    school_boundaries = school_boundaries.drop(columns=['index_right'])
    school_boundaries = school_boundaries.sort_values(['school_name']).reset_index(drop=True)
    return gpd.GeoDataFrame(school_boundaries, geometry='geometry', crs='EPSG:3414')

school_polygons = attach_school_boundaries(gdf_schools, LAND_USE_SOURCES, land_use_layers)

print(school_polygons[['school_name', 'land_use_source', 'LU_DESC']].head())


In [ ]:
school_polygons=school_polygons[['school_name', 'postal_code', 'X', 'Y', 'geometry', 'start_year', 'end_year']]

In [ ]:
school_polygons.dropna(subset=['X','Y'], inplace=True)

In [ ]:
from shapely.geometry import Point
import pandas as pd

missing_mask = school_polygons['geometry'].isna()

for source_name in reversed(list(land_use_layers.keys())):
    layer = land_use_layers[source_name].copy()

    if layer.crs.to_epsg() != 3414:
        layer = layer.to_crs(epsg=3414)

    missing_idx = school_polygons.index[missing_mask]
    if len(missing_idx) == 0:
        break

    for idx in missing_idx:
        x = school_polygons.at[idx, 'X']
        y = school_polygons.at[idx, 'Y']

        if pd.isna(x) or pd.isna(y):
            continue

        pt = Point(float(x), float(y))

        # first try exact match
        match = layer[layer.covers(pt)]

        if match.empty:
            distances = layer.geometry.distance(pt)
            nearest_idx = distances.idxmin()
            nearest_dist = distances.loc[nearest_idx]

            if nearest_dist <= 1.0:   # 1 metre tolerance; for Temasek Primary School
                match = layer.loc[[nearest_idx]]

        if not match.empty:
            school_polygons.at[idx, 'geometry'] = match.iloc[0]['geometry']

    missing_mask = school_polygons['geometry'].isna()

Temasek Primary School X and Y is very near an educational instuition plot, but not within it. Just for taking note.

In [ ]:
school_polygons[school_polygons.isna().any(axis=1)]

In [ ]:
school_polygons.to_file(
    "../data/processed/schools/final_primary_schools_with_school_boundaries.geojson",
    driver="GeoJSON"
)

In [ ]:
school_boundaries_geojson = school_polygons.drop(columns=["school_point"]).copy()

school_boundaries_geojson = gpd.GeoDataFrame(
    school_boundaries_geojson,
    geometry="geometry",
    crs="EPSG:3414"
)

school_boundaries_geojson.to_file(
    "../data/processed/schools/final_pri_schs_with_school_boundaries.geojson",
    driver="GeoJSON"
)

## Alternative Boundary Matching Pass
This later block appears to be a second pass or refinement of the boundary-construction workflow. It repeats the polygon-matching logic with additional diagnostics before writing the final GeoJSON outputs.


## 13/3 - Adding boundaries to the school, instead of the center

In [ ]:
import geopandas as gpd
from pathlib import Path

# def resolve_data_path(relative_path: str) -> Path:
#     candidates = [Path(relative_path), Path('..') / relative_path]
#     for path in candidates:
#         if path.exists():
#             return path
#     raise FileNotFoundError(f'Could not find {relative_path}')

def load_education_land_use(path: Path) -> gpd.GeoDataFrame:
    land_use = gpd.read_file(path)
    if land_use.crs is None or land_use.crs.to_epsg() != 3414:
        land_use = land_use.to_crs(3414) # ensure consistent CRS for spatial operations (3414 is EPSG:XY used in Singapore)
    land_use = land_use[land_use['LU_DESC'].str.contains('EDUCATION', case=False, na=False)].copy() # filter to educational institutions only
    land_use = land_use.reset_index(drop=True).reset_index(names='land_use_index')
    return land_use

LAND_USE_SOURCES = {
    'mp2014': {
        'path': '../data/raw/land_use/2014/G_MP14_LAND_USE_PL.shp',
        'years': range(2014, 2019),
    },
    'mp2019': {
        'path': '../data/raw/land_use/MasterPlan2019LandUselayer.geojson',
        'years': range(2019, 2025),
    },
    'mp2025': {
        'path': '../data/raw/land_use/MasterPlan2025LandUseLayer.geojson',
        'years': range(2025, 2027),
    },
}

land_use_layers = {
    source_name: load_education_land_use(config['path'])
    for source_name, config in LAND_USE_SOURCES.items()
}

for source_name, land_use_layer in land_use_layers.items():
    print(source_name, land_use_layer.shape)


In [ ]:
import pandas as pd

# Read primary schools data and keep only years covered by the land-use layers
pri_schs_df = pd.read_csv('../data/processed/schools/final_primary_schools.csv')

valid_years = sorted({
    year
    for config in LAND_USE_SOURCES.values()
    for year in config['years']
})
pri_schs_df = pri_schs_df[pri_schs_df['year'].isin(valid_years)].copy()

pri_schs_df.head()


In [ ]:
gdf_schools = gpd.GeoDataFrame(
    pri_schs_df,
    geometry=gpd.points_from_xy(pri_schs_df['X'], pri_schs_df['Y']),
    crs="EPSG:3414"
)

def attach_school_boundaries(
    schools: gpd.GeoDataFrame,
    land_use_sources: dict,
    land_use_layers: dict,
) -> gpd.GeoDataFrame:
    matched_parts = []

    for source_name, config in land_use_sources.items():
        school_subset = schools[schools['year'].isin(config['years'])].copy()
        school_subset['land_use_source'] = source_name

        joined = school_subset.sjoin(
            land_use_layers[source_name][['land_use_index', 'LU_DESC', 'geometry']],
            how='left',
            predicate='within'
        ).rename(columns={'geometry': 'school_point'})

        joined = joined.merge(
            land_use_layers[source_name][['land_use_index', 'geometry']],
            on='land_use_index',
            how='left'
        )

        matched_parts.append(joined)

    school_boundaries = pd.concat(matched_parts, ignore_index=True)
    school_boundaries = school_boundaries.drop(columns=['index_right'])
    school_boundaries = school_boundaries.sort_values(['year', 'school_name']).reset_index(drop=True)
    return gpd.GeoDataFrame(school_boundaries, geometry='geometry', crs='EPSG:3414')

school_polygons = attach_school_boundaries(gdf_schools, LAND_USE_SOURCES, land_use_layers)

print(school_polygons[['school_name', 'year', 'land_use_source', 'LU_DESC']].head())


In [ ]:
school_polygons[['school_name', 'year', 'land_use_source', 'LU_DESC', 'school_point', 'geometry']].head()


In [ ]:
boundary_match_summary = (
    school_polygons.assign(has_boundary=school_polygons.geometry.notna())
    .groupby(['land_use_source', 'year'])['has_boundary']
    .agg(matched='sum', total='size')
)
boundary_match_summary['unmatched'] = boundary_match_summary['total'] - boundary_match_summary['matched']
boundary_match_summary


In [ ]:
school_polygons[school_polygons.geometry.isna()][
    ['school_name', 'year', 'land_use_source', 'X', 'Y']
].reset_index(drop=True).head(20)


In [ ]:
# recheck through mp2019 
mp2014_unmatched = school_polygons[
    (school_polygons['land_use_source'] == 'mp2014') &
    (school_polygons.geometry.isna())
].copy()

mp2014_unmatched_points = gpd.GeoDataFrame(
    mp2014_unmatched.drop(columns=['LU_DESC', 'land_use_index', 'geometry'], errors='ignore'),
    geometry='school_point',
    crs='EPSG:3414'
)

mp2014_unmatched_matched_to_mp2019 = mp2014_unmatched_points.sjoin(
    land_use_layers['mp2019'][['land_use_index', 'LU_DESC', 'geometry']],
    how='left',
    predicate='within'
).rename(columns={'geometry': 'school_point'})

mp2014_unmatched_matched_to_mp2019 = mp2014_unmatched_matched_to_mp2019.merge(
    land_use_layers['mp2019'][['land_use_index', 'geometry']],
    on='land_use_index',
    how='left'
)

mp2014_unmatched_matched_to_mp2019[['school_name', 'year', 'LU_DESC', 'geometry']].head()

In [ ]:
school_polygons.to_file(
    "../data/processed/schools/final_primary_schools_with_school_boundaries.geojson",
    driver="GeoJSON"
)

In [ ]:
school_boundaries_geojson = school_polygons.drop(columns=["school_point"]).copy()

school_boundaries_geojson = gpd.GeoDataFrame(
    school_boundaries_geojson,
    geometry="geometry",
    crs="EPSG:3414"
)

school_boundaries_geojson.to_file(
    "../data/processed/schools/final_pri_schs_with_school_boundaries.geojson",
    driver="GeoJSON"
)